# 03.07 - Orientation + CV Role

**Notebook type:** Solution notebook with full working answers.

**Daily output:** a CV task checklist you can reuse during OLP AI contests.

Today is not about writing a model yet. It is about becoming the teammate who can look at a CV problem and quickly answer: what data do we have, what should the model output, what metric matters, how do we validate, and what can go wrong?


## Contest Workflow Mental Model

A CV pipeline usually has these stages:

1. Understand task statement and metric.
2. Inspect train/test files.
3. Build label mapping.
4. Build image/video readers.
5. Create train/validation split.
6. Train a baseline.
7. Evaluate with the real contest metric.
8. Analyze errors.
9. Run inference on test data.
10. Create and validate submission.

As CV lead, your job is not only accuracy. You also protect the team from format mistakes, path bugs, label-order bugs, slow inference, and invalid submissions.


## Key Concepts

**Input format:** Where are the files? Are examples images, videos, folders, CSV rows, or JSON records?

**Label mapping:** Human labels must become integer IDs for training, then convert back for submission.

**Validation:** Local validation must imitate the leaderboard metric. If the metric is Macro-F1, optimize and report Macro-F1, not only accuracy.

**Inference speed:** A strong model that times out is not a valid contest solution.

**Submission contract:** Row count, ID order, label names, column names, and missing values must be checked before submitting.


In [ ]:
task_summary = {
    "task_type": "image or video classification",
    "input_files": [
        "train.csv with image_path/video_path and label",
        "test.csv with id and image_path/video_path",
        "train image/video folder",
        "sample_submission.csv",
    ],
    "target_column": "label",
    "id_column": "id",
    "metric": "Macro-F1",
    "submission_columns": ["id", "label"],
    "main_risks": [
        "wrong label_to_id order",
        "RGB/BGR mismatch",
        "corrupt image or unreadable video",
        "validation split leakage",
        "test row order mismatch",
        "inference timeout",
    ],
}

for key, value in task_summary.items():
    print(f"{key}: {value}")


## Macro-F1 Refresher

Macro-F1 computes F1 for each class, then averages them equally. It punishes models that ignore minority classes.

For each class:

- precision = true positives / predicted positives
- recall = true positives / actual positives
- F1 = harmonic mean of precision and recall

If the contest metric is Macro-F1, a model that only predicts the majority class can look decent by accuracy but fail the actual goal.


In [ ]:
def macro_f1_report(y_true, y_pred, num_classes):
    rows = []
    for c in range(num_classes):
        tp = sum((yt == c and yp == c) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != c and yp == c) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == c and yp != c) for yt, yp in zip(y_true, y_pred))
        support = sum(yt == c for yt in y_true)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            "class": c,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        })
    macro_f1 = sum(row["f1"] for row in rows) / num_classes
    return rows, macro_f1

y_true = [0, 0, 1, 1, 2, 2]
y_pred = [0, 0, 0, 1, 1, 1]
rows, macro = macro_f1_report(y_true, y_pred, 3)
for row in rows:
    print(row)
print("macro_f1:", round(macro, 4))


## CV Lead Checklist

Build a checklist that catches the common contest failures:

- data files and paths
- label mapping
- image/video reading
- train/validation split
- metric
- model baseline
- inference
- submission file
- reproducibility notes


In [ ]:
cv_task_checklist = {
    "data": [
        "Confirm train/test CSV paths exist.",
        "Count number of training and test examples.",
        "Check file extensions and folder structure.",
        "Identify missing, duplicate, or corrupt files.",
    ],
    "labels": [
        "List unique labels and class counts.",
        "Create stable label_to_id and id_to_label mappings.",
        "Save mappings with checkpoints and submissions.",
        "Check that labels in validation and test submission use expected names.",
    ],
    "reading": [
        "For images, verify RGB/BGR convention.",
        "For grayscale/RGBA images, force a consistent RGB contract.",
        "For videos, check FPS, frame count, failed decodes, and sampling policy.",
    ],
    "validation": [
        "Use a stratified split for classification when possible.",
        "Avoid leakage from duplicate files, same video, same subject, or same scene.",
        "Keep validation transforms deterministic.",
    ],
    "metric": [
        "Compute the exact contest metric locally.",
        "For Macro-F1, inspect per-class precision, recall, and F1.",
        "Track metric on validation, not training data.",
    ],
    "baseline": [
        "Start with the simplest working model.",
        "Record image size, transforms, model name, learning rate, epochs, and seed.",
        "Save the best checkpoint by validation metric.",
    ],
    "inference": [
        "Use model.eval() and no_grad/inference_mode.",
        "Batch test examples when memory allows.",
        "Measure runtime on a representative subset.",
    ],
    "submission": [
        "Match sample submission column names.",
        "Match test IDs and row count exactly.",
        "Check for NaN, empty labels, duplicate IDs, and invalid class names.",
    ],
    "reproducibility": [
        "Set seeds.",
        "Keep config notes beside checkpoint.",
        "Write down package versions when results matter.",
    ],
}

for section, checks in cv_task_checklist.items():
    print(section.upper())
    for check in checks:
        print("-", check)
